Business Problem:

- The use case we have in here to create a brochure by giving the website name and letting the LLM/Model figure out what needs to be done

The thought process is:
1. We take a website. Then we understand/scrap all its links and find the most relevant links like About Us, Careers, Products with a help of a Model
2. We then take those urls and scrap those web pages and then with a help of a model we create a Marketing Brochure

In [34]:
#Step 1: Import all necessary utilities

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import fetch_website_contents, fetch_website_links
import json

In [3]:
#Step 2: Have our key up and running
load_dotenv(override=True)
open_api_key = os.getenv('OPENAI_API_KEY')

if not open_api_key:
    print('No API Key found')
elif not open_api_key.startswith('sk-proj'):
    print('API key details are not correct')
else:
    print('API key found')

model = 'gpt-5-nano'
openai = OpenAI()

API key found


In [10]:
#Step 3: Have our function scrap all the important links from the passed website in the definition

links = fetch_website_links('https://huggingface.co/')
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.5-35B-A3B',
 '/Qwen/Qwen3.5-9B',
 '/unsloth/Qwen3.5-35B-A3B-GGUF',
 '/Qwen/Qwen3.5-27B',
 '/Qwen/Qwen3.5-0.8B',
 '/models',
 '/spaces/FrameAI4687/Omni-Video-Factory',
 '/spaces/multimodalart/qwen-image-multiple-angles-3d-camera',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/mrfakename/Z-Image-Turbo',
 '/spaces/HuggingFaceM4/faster-qwen3-tts-demo',
 '/spaces',
 '/datasets/peteromallet/dataclaw-peteromallet',
 '/datasets/togethercomputer/CoderForge-Preview',
 '/datasets/nohurry/Opus-4.6-Reasoning-3000x-filtered',
 '/datasets/TuringEnterprises/Open-RL',
 '/datasets/crownelius/Opus-4.6-Reasoning-3300x',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook',
 '

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [20]:
def get_user_prompt(url):
	user_prompt = f"""" 
	Here is the links to the website {url}
	Please decide which of these are relevant to the make the brochure of the company.
	Respond with the JSON format. Do not include Terms of Service, Privacy , email, links
	"""
	
	links = fetch_website_links(url)
	user_prompt += "\n".join(links)
	return user_prompt

In [21]:
def get_relevant_links(url):
    response = openai.chat.completions.create(
        model = model,
        messages = [
            {"role" : "system",
            "content" : link_system_prompt},
            {"role" : "user",
            "content" : get_user_prompt(url)}
        ] 
    )
    results = response.choices[0].message.content
    links = json.loads(results)
    return links

In [22]:
get_relevant_links('https://www.revaine.nl/')

{'links': [{'type': 'home page', 'url': 'https://www.revaine.nl/'},
  {'type': 'linkedin company page',
   'url': 'https://www.linkedin.com/company/106537257'}]}

In [23]:
def get_relevant_links_modified(url):
    print(f'Getting the relevant links for website:{url} via the model : {model}')
    response = openai.chat.completions.create(
        model = model,
        messages=[
            {"role" : "system",
            "content" : link_system_prompt},
            {"role" : "user",
            "content" : get_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    results = response.choices[0].message.content
    links = json.loads(results)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [24]:
get_relevant_links_modified('https://www.revaine.nl/')

Getting the relevant links for website:https://www.revaine.nl/ via the model : gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://www.revaine.nl'},
  {'type': 'linkedin company page',
   'url': 'https://www.linkedin.com/company/106537257'}]}

In [27]:
#Step 4: Now lets build a brochure

def fetch_page_and_all_details(url):
    contents = fetch_website_contents(url)
    relevant_links = get_relevant_links_modified(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [28]:
fetch_page_and_all_details('https://www.revaine.nl/')

Getting the relevant links for website:https://www.revaine.nl/ via the model : gpt-5-nano
Found 2 relevant links


'## Landing Page:\n\nHome | revaine.nl\n\ntop of page\nHome\nWhy revaine\nOur Services\nOur Approach\nAbout Us\nOur Values\nContact Us\nMore\nUse tab to navigate through the menu items.\nTrusted partner in Revenue Life Cycle Optimization\nWe are on a\u200b mission to simplify the Lead2Revenue journey.\n\u200b\nWe have spent over a decade helping enterprises succeed with Salesforce and Conga. Our strength lies in making complex data integrations and RLM projects simple and effective — bringing together data architects, developers, and project managers to deliver results that last in Manufacturing, Healthcare, Cosmetics , Automobile and Pharma industries.\n\u200b\nWe are recognized by World Leading Platforms and Solutions like Conga, Salesforce, and DevRev\n\u200b\nTransform\nLead-to-Revenue\xa0 with Seemless Data Integration and AI\nBook a Free Consultation\nWhy revaine?\nSystem-Agnostic Lead-to-Revenue Expertise\nProven Expertise & Strategic Partnerships\nFuture-Proof Revenue Operation

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt("Revaine.nl", 'https://www.revaine.nl/')

Getting the relevant links for website:https://www.revaine.nl/ via the model : gpt-5-nano
Found 2 relevant links


'\nYou are looking at a company called: Revaine.nl\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome | revaine.nl\n\ntop of page\nHome\nWhy revaine\nOur Services\nOur Approach\nAbout Us\nOur Values\nContact Us\nMore\nUse tab to navigate through the menu items.\nTrusted partner in Revenue Life Cycle Optimization\nWe are on a\u200b mission to simplify the Lead2Revenue journey.\n\u200b\nWe have spent over a decade helping enterprises succeed with Salesforce and Conga. Our strength lies in making complex data integrations and RLM projects simple and effective — bringing together data architects, developers, and project managers to deliver results that last in Manufacturing, Healthcare, Cosmetics , Automobile and Pharma industries.\n\u200b\nWe are recognized by World Leading Platforms and Solutions like Conga, Salesforce, and DevRev\n\u200b\nTransf

In [31]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

create_brochure("Revaine.nl", 'https://www.revaine.nl/')

Getting the relevant links for website:https://www.revaine.nl/ via the model : gpt-5-nano
Found 2 relevant links


# Revaine.nl - Simplifying Your Lead-to-Revenue Journey

---

## About Revaine

Revaine.nl is a trusted boutique consultancy focused on Revenue Life Cycle Optimization, dedicated to simplifying the Lead-to-Revenue (L2R) journey for enterprises. With over a decade of experience, Revaine combines deep expertise with strategic partnerships to deliver seamless, AI-powered, and system-agnostic solutions.

We specialize in transforming complex data integrations and Revenue Life Cycle Management projects by harmonizing the skills of data architects, developers, and project managers. Our work spans multiple industries including Manufacturing, Healthcare, Cosmetics, Automobile, and Pharmaceuticals.

Recognized by industry leaders like Conga, Salesforce, and DevRev, Revaine is a partner committed to innovation and excellence in revenue operations.

---

## Why Choose Revaine?

- **System-Agnostic Expertise:** We design and implement solutions that work across all major platforms.
- **Proven Partnerships:** Trusted by leading technology providers ensuring cutting-edge solutions.
- **AI-Enabled Revenue Operations:** Future-proof your business by leveraging artificial intelligence to optimize your sales and finance processes.
- **Human-First Transformation:** Our consultants combine industry experience with a people-centered approach to change management.
- **Tangible Business Outcomes:**  
  - Shorten lead-to-revenue cycles by up to 40%  
  - Increase sales efficiency by 20-30%  
  - Improve revenue forecasting accuracy to 90%

We enable businesses to cut out manual work and complexity so that more energy is focused on creating customer value rather than administrative processes.

---

## Our Services

### Advisory Services  
Strategic advisory for AI-driven L2R transformation and adoption. We help design future-proof revenue strategies and guide organizational change to ensure smooth implementation and successful, lasting adoption.

### Implementation & Transformation  
Comprehensive end-to-end system integration and AI-powered transformation projects, helping to streamline sales, finance, service, supply chain, and revenue workflows.

### Expert Staffing  
Providing skilled professionals including data architects, developers, and project managers, ensuring your projects succeed with the right expertise.

---

## Industries Served

- Manufacturing
- Healthcare
- Cosmetics
- Automobile
- Pharmaceuticals

---

## Company Culture & Values

At Revaine, we believe in a **human-first approach** to technology transformation. Our culture fosters collaboration between expert consultants and client teams, emphasizing simplicity, innovation, and long-term impact. We value:

- Integrity and transparency in our partnerships
- Continuous learning and adapting to emerging technologies
- Delivering measurable business results through tailored solutions
- Supporting employee growth and expertise development

---

## Careers at Revaine

Join a dynamic team committed to driving digital transformations that matter. If you are passionate about AI, CRM systems like Salesforce, and want to work in an industry-diverse, innovative consultancy environment, Revaine is an excellent place to grow your career.

We seek talented professionals such as:

- Data Architects
- Software Developers
- Project Managers
- AI and L2R Consultants

Leverage your skills with us to simplify complex processes and deliver impactful results for global clients.

---

## Contact Us

**Ready to transform your Lead-to-Revenue process?**

Book a free consultation to explore how Revaine can help simplify your revenue journey with AI-powered, seamless integration solutions.

Website: [revaine.nl](https://revaine.nl)  
Email: contact@revaine.nl  
Phone: +31 (Your local number)

---

**Revaine.nl** — Your trusted partner in Lead-to-Revenue Transformation with AI and seamless system integration.

In [35]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [36]:
stream_brochure("Revaine.nl", 'https://www.revaine.nl/')

Getting the relevant links for website:https://www.revaine.nl/ via the model : gpt-5-nano
Found 2 relevant links


# Revaine.nl Brochure

---

## About Revaine

Revaine is a trusted boutique consultancy specializing in **Revenue Life Cycle Optimization** with a mission to simplify the Lead-to-Revenue (L2R) journey. Over the past decade, Revaine has partnered with enterprises to successfully implement and optimize Salesforce and Conga platforms, delivering seamless data integration and AI-powered transformations.

Headquartered in the Netherlands, Revaine serves a diverse range of industries including **Manufacturing, Healthcare, Cosmetics, Automobile, and Pharma**. Recognized by industry leaders such as Conga, Salesforce, and DevRev, Revaine brings unparalleled expertise in making complex relationship lifecycle management (RLM) and data integration projects simple and effective.

---

## What We Do

### Lead-to-Revenue Transformation Services

- **Strategic Advisory Services:** Crafting future-proof L2R strategies with AI integration for optimized sales, finance, and revenue operations. We emphasize structured change management to ensure smooth adoption across organizations.
  
- **Implementation & Transformation:** Providing end-to-end system integrations that connect multiple platforms seamlessly, simplifying workflows across supply chain, sales, service, finance, and revenue departments.

- **Expert Staffing:** Offering specialized consultant teams including data architects, developers, and project managers to deliver impactful and lasting results.

---

## Why Choose Revaine?

- **System-Agnostic Expertise:** We work across platforms, ensuring flexibility and optimized solutions tailored to your systems.
  
- **Proven Industry Expertise:** Deep experience with Salesforce, Conga, and AI-driven technologies across various industries.
  
- **AI-Powered, Human-First Approach:** Combining sophisticated AI tools with human insight to future-proof your revenue operations.
  
- **Tangible Results:**  
  - Shorten Lead-to-Revenue cycles by up to **40%**  
  - Increase sales efficiency by **20-30%**  
  - Improve revenue forecasting accuracy to **90%**

- **Customer-Centric Focus:** Our methodology reduces manual work and complexity, so your teams can spend more time creating value for your customers.

---

## Our Company Culture & Values

Revaine prides itself on fostering a culture that values **innovation, collaboration, and continuous learning**. Our consultants bring immense industry experience and work closely with clients to understand their unique challenges and tailor solutions accordingly.

We believe in a **human-first transformation**—balancing advanced AI solutions with personal touch and strategic vision. Our team thrives in an environment where every member contributes meaningfully toward client success and operational excellence.

---

## Industries We Serve

- Manufacturing  
- Healthcare  
- Cosmetics  
- Automobile  
- Pharmaceuticals  

Because we understand the distinct revenue lifecycle challenges in these industries, we deliver customized, impactful solutions that help clients gain competitive advantage.

---

## Careers at Revaine

Revaine is continuously seeking **talented professionals** passionate about technology, AI, and revenue operations. If you are interested in joining a dynamic team that values expertise and innovation, and working on transformative projects across leading global platforms, Revaine may be the ideal place for you.

We offer opportunities in roles including:  
- Data Architects  
- Salesforce and Conga Developers  
- Project Managers  
- Strategic Consultants  

Join us to help reimagine the Lead-to-Revenue experience for enterprises worldwide.

---

## Contact Us

Ready to transform your revenue lifecycle? Book a free consultation or get in touch to discuss how Revaine can help your business thrive.

Website: [www.revaine.nl](https://www.revaine.nl)  
Email: contact@revaine.nl  
Phone: +31 (0)...

---

**Revaine** – Your Trusted Partner in Lead-to-Revenue Transformation and AI-Driven Revenue Optimization.